In [ ]:
import sys 
sys.path.append('/Users/amalyajohnson/Desktop/SpectraScope/spectrascope_local/')
%load_ext autoreload
%autoreload 2

from spectrascope.SpectraScope import *
from spectrascope.features import * 
from sklearn.linear_model import Ridge, Lasso, ElasticNet
import SPA 
import os
from mat4py import loadmat

In [ ]:
spec_dir = '/Users/amalyajohnson/Desktop/SpectraScope/Datasets/Grape-Chemical-Composition-main/Spectroscopic data (Matlab files)/'
true_dir = '/Users/amalyajohnson/Desktop/SpectraScope/Datasets/Grape-Chemical-Composition-main/Ground Truth/'

In [ ]:
from mat4py import loadmat

fn = spec_dir + 'RAMAN_Data_09082022.mat'
fn2 = spec_dir + 'NIR_Data_09082022.mat'

wls_nir = []
wls_rm = []
shifts = []
names_nir = []
names_rm = []
data_nir = {}
data_rm = {}
for fn in os.listdir(spec_dir):
    if 'Data.mat' not in fn: 
        data = loadmat(spec_dir + fn)
        name = fn.split('/')[-1].split('.')[0]
        wl = np.array(data[name]['wvlength'])
    else: 
        continue
    if 'NIR' in name:
        names_nir.append(name)
        wls_nir.append(wl)
        for key2 in data[name]['spec']: 
            arr = data[name]['spec'][key2]
            arr = np.array(arr)
            data_nir[name + '_' + key2] = arr
    elif 'RAMAN' in name: 
        shift = data[name]['raman_shift']
        shifts.append(shift)
        names_rm.append(name) 
        wls_rm.append(wl)
        for key1 in data[name]['raman']: 
            for key2 in data[name]['raman'][key1]: 
                arr = data[name]['raman'][key1][key2]
                arr = np.array(arr)
                if key1 == 'Cab':
                    data_rm[name + '_' + key1 + '_' + key2] = arr 

In [496]:
data = loadmat(spec_dir + 'specData.mat')
XSC = np.array(data['XSC'])
XSF = np.array(data['XSF'])

data = loadmat(spec_dir + 'ramanData.mat')
XRC = np.array(data['XRC'])
XRF = np.array(data['XRF'])

data = loadmat(true_dir + 'trueData.mat')
YF = np.array(data['YF'])
YC = np.array(data['YC']) 

In [497]:
shiftsi = np.array([(int(i), s[0]) for i,s in enumerate(shifts[0]) if str(s[0]) != 'nan'])
shifts_ids = shiftsi[:,0]
raman_shifts = shiftsi[:,1]

wlsi = np.array([(int(i), s[0]) for i,s in enumerate(wls_rm[0]) if str(s[0]) != 'nan'])
wls_ids = wlsi[:,0]
raman_wls = wlsi[:,1]

nir_wls = np.array(wls_nir[0])
nir_wls = nir_wls.reshape(len(nir_wls),)

In [498]:
XRC = XRC[:,np.array([int(i) for i in wls_ids])]
XRF = XRF[:,np.array([int(i) for i in wls_ids])]

In [ ]:
fig = make_subplots(1,2)
fig.add_trace(go.Scatter(x=nir_wls, y=XSC[0]), row=1, col=1)
fig.add_trace(go.Scatter(x = raman_wls, y = XRC[0]), row=1, col=2)

fig.add_trace(go.Scatter(x=nir_wls, y=XSF[0]), row=1, col=1)
fig.add_trace(go.Scatter(x = raman_wls, y = XRF[0]), row=1, col=2)

fig.show()

In [ ]:
# Deprecated ; previous loading data 
shifts = np.array(shifts)

Fph = white['pH'].to_numpy()[white['pH'].isna().to_numpy() == False]
FBrix = white['Brix'].to_numpy()[white['Brix'].isna().to_numpy() == False]

Cph = red['pH'].to_numpy()[red['pH'].isna().to_numpy() == False]
CBrix = red['Brix'].to_numpy()[red['pH'].isna().to_numpy() == False]

XC_nir = data_nir['NIR_Data_07292022_Cab']
XF_nir = data_nir['NIR_Data_07292022_French'][:-1][[white['pH'].isna().to_numpy() == False][0][:148]]

XC_rm = data_rm['RAMAN_Data_07292022_Cab1_blr']
XF_rm = data_rm['RAMAN_Data_07292022_French_blr'][[white['pH'].isna().to_numpy() == False][0][:148]]

wlsi = np.array([(int(i), s[0]) for i,s in enumerate(wls[0]) if str(s[0]) != 'nan'])
ids = wlsi[:,0]
wlsi = wlsi[:,1]
XF_nir = XF_nir[:,np.array([int(i) for i in ids])]

In [685]:
#Cleaning up NaNs 
def lose_nans(X, y): 
    ids = np.argwhere(np.isnan(y) == False).ravel()
    y = y[ids]
    X = X[ids]
    # true for all? 
    y = np.insert(y, 29, np.nan)
    nsamples = len(y) // 6
    y = y.reshape(nsamples,6)
    y = y[:,0]
    
    X = np.insert(X.astype(float), 29, np.nan, axis=0)
    X = X.reshape(nsamples,6,X.shape[1])
    X = np.nanmean(X, axis=1)
    return X,y 

XSF_ph, ph = lose_nans(XSF, YF[:,0])
XRF_ph, ph = lose_nans(XRF, YF[:,0])

XSF_br, br = lose_nans(XSF, YF[:,1])
XRF_br, br = lose_nans(XRF, YF[:,1])

XSF_ta, ta = lose_nans(XSF, YF[:,2])
XRF_ta, ta = lose_nans(XRF, YF[:,2])

In [720]:
XF = [XSF_br, XRF_br]

ss_f = SpectraScope(XF, br, specnames=['nir', 'rm'], yname='ph', random_seed=10)
ss_f.add_x_axis('rm', raman_wls)
ss_f.add_x_axis('nir', nir_wls)

saving to /examples/models, change save_path kwarg if youd like to change


In [721]:
ss_f.add_spectral_feature('rm', 'pointwise_features', xrange=ss_f.rm_xaxis)
ss_f.add_spectral_feature('rm', 'polynomial_featuresall')
ss_f.add_spectral_feature('rm', 'featureagg', n_clusters=15)
ss_f.add_spectral_feature('rm', 'pca', n_components=15)
ss_f.add_spectral_feature('rm', 'cdf')
ss_f.add_spectral_feature('rm', 'cwt')

ss_f.add_spectral_feature('nir', 'pointwise_features', xrange=ss_f.nir_xaxis)
ss_f.add_spectral_feature('nir', 'polynomial_featuresall')
ss_f.add_spectral_feature('nir', 'featureagg', n_clusters=15)
ss_f.add_spectral_feature('nir', 'pca', n_components=15)
ss_f.add_spectral_feature('nir', 'cdf')
ss_f.add_spectral_feature('nir', 'cwt')

In [722]:
ss_f.transform_feature('rm_pointwise_features', degree=2, interaction=False)
ss_f.transform_feature('nir_pointwise_features', degree=2, interaction=False)

ss_f.combine_feature('rm_pointwise_features', 'nir_pointwise_features')
# ss_f.transform_feature('rm_nir_pointwise_features', degree=1, interaction=True)

In [957]:
RF_kwargs = {'estimator':RandomForestRegressor(), 
                'param_grid':{'n_estimators':[200], 
                                        'criterion':['squared_error'],
                                        'max_depth': [50, None]},
                'refit':True,
                'metric':mean_squared_error}

lcen_kwargs = {'l1_ratio':[0.001,0.005, 0.01, 0.1,0.3], 
               'K_fold':3, 
               'cv_method':'Kfold', 
               'alpha':5, 
               'degree':[1],
               'trans_type':'poly', 
               'LCEN_cutoff':5e-2, 
               'LCEN_interaction':False, 
               'scale_X':True, 
               'scale_y':True,
               'metric':root_mean_squared_error}


Lasso_kwargs = {'estimator':Lasso()}

In [727]:
ss_f.add_y('br', br)

In [953]:
feats = ['pointwise_features', 'cdf', 'featureagg', 'polynomial_featuresall', 'pca', 'pointwise_features_trans',]
from sklearn.metrics import root_mean_squared_error
for spec in ss_f.specnames:
    for feat in feats: 
        attr_name = f'{spec}_{feat}'
        ss_f.do_sk(attr_name, 'br', 0.25, use_pipeline=False, estimator=RandomForestRegressor(), metric=root_mean_squared_error)

ss_f.do_sk('rm_nir_pointwise_features', 'br', 0.25, use_pipeline=False, estimator=RandomForestRegressor(), metric=root_mean_squared_error)

Model is not fitted yet.
Model is not fitted yet.
Model is not fitted yet.
Model is not fitted yet.
Model is not fitted yet.
Model is not fitted yet.
Model is not fitted yet.
Model is not fitted yet.
Model is not fitted yet.
Model is not fitted yet.
Model is not fitted yet.
Model is not fitted yet.
Model is not fitted yet.


In [955]:
ss_f.plot_df_heatmap(pd.DataFrame(ss_f.best_fits).T)

In [958]:
feats_lcen = ['pointwise_features', 'cdf', 'featureagg', 'polynomial_featuresall', 'pca']

for spec in ss_f.specnames:
    for feat in feats_lcen: 
        attr_name = f'{spec}_{feat}'
        ss_f.do_LCEN(attr_name, 'br', 0.25, **lcen_kwargs)

ss_f.do_LCEN('rm_nir_pointwise_features', 'br', 0.25, **lcen_kwargs)

Beginning variable selection runs. There are 6 hyperparameter combinations
Beginning real runs. There are 30 hyperparameter combinations
Completed model LCENf fold   3
=== Residual Analysis ===
RMSE = 1.5612279764170862
APE = 7.701106569111831
The best model is LCEN. View its results via fitting_result["LCEN"] or by opening the SPA_results json/pickle files.
Beginning variable selection runs. There are 6 hyperparameter combinations
Beginning real runs. There are 30 hyperparameter combinations
Completed model LCENf fold   3
=== Residual Analysis ===
RMSE = 1.8325091229327435
APE = 8.501158685684175
The best model is LCEN. View its results via fitting_result["LCEN"] or by opening the SPA_results json/pickle files.
Beginning variable selection runs. There are 6 hyperparameter combinations
Beginning real runs. There are 30 hyperparameter combinations
Completed model LCENf fold   3
=== Residual Analysis ===
RMSE = 1.474640406392602
APE = 6.8198474790875245
The best model is LCEN. View its r

In [980]:
cols = [c for c in pd.DataFrame(ss_f.best_fits).T.columns if 'ph' not in c]
temp = pd.DataFrame(pd.DataFrame(ss_f.best_fits)[['LCEN', 'RandomForestRegressor']]).dropna().T

In [1001]:
ss_f.plot_df_heatmap(100 * temp/np.max(br))

In [756]:
to_plot = feats #['pointwise_features', 'cdf', 'featureagg', 'polynomial_featuresall', 'pca']
fig = make_subplots(1,6, subplot_titles=to_plot)

ytrue = ss_f.ph_test
for i,feat in enumerate(to_plot):
    model = getattr(ss_f, f'rm_{feat}_ph_RandomForestRegressor')
    ypred = model['ypred']
    score = 100 * model['score'] / ss_f.ph.max()
    score = model['score']
    fig.add_trace(go.Scatter(x=ytrue, y=ypred, mode='markers', name=f'raman, RMSE={round(score,4)} '), row=1, col=i+1)
    
    model = getattr(ss_f, f'nir_{feat}_ph_RandomForestRegressor')
    ypred = model['ypred']
    score = 100 * model['score'] / ss_f.ph.max()
    score = model['score']
    fig.add_trace(go.Scatter(x=ytrue, y=ypred, mode='markers', name=f'nir, RMSE={round(score,4)} '), row=1, col=i+1)
    
    fig.add_trace(go.Scatter(x=np.arange(ytrue.min(), ytrue.max(), 0.1), y=np.arange(ytrue.min(), ytrue.max(), 0.1), mode='lines', marker=dict(color='black'), showlegend=False), row=1, col=i+1)
    fig.update_xaxes(title_text='True', row=1, col=i+1)
    fig.update_yaxes(title_text='Predicted', row=1, col=1)
fig.update_layout(title_text='Predicting Brixs, French')

In [819]:
from spectrascope.features import SpectralFeatures

In [1047]:
ss_f.best_fits

{'RandomForestRegressor': {'nir_pointwise_features_ph': 1.8629997378571501,
  'nir_cdf_ph': 2.031956457828666,
  'nir_featureagg_ph': 1.6926844511493548,
  'nir_polynomial_featuresall_ph': 1.4346714557857876,
  'nir_pca_ph': 1.4232258478377517,
  'nir_pointwise_features_trans_ph': 2.3874495781842366,
  'rm_pointwise_features_ph': 2.224912713208173,
  'rm_cdf_ph': 2.454172626175626,
  'rm_featureagg_ph': 2.1965037969993464,
  'rm_polynomial_featuresall_ph': 1.7057471935700483,
  'rm_pca_ph': 1.8919510355530431,
  'rm_pointwise_features_trans_ph': 2.231401608750627,
  'rm_nir_pointwise_features_ph': 1.8850357716672421,
  'nir_pointwise_features_br': 1.8554405168659835,
  'nir_cdf_br': 2.013085899113422,
  'nir_featureagg_br': 1.7572293201858815,
  'nir_polynomial_featuresall_br': 1.4252297244451013,
  'nir_pca_br': 1.4124483855562526,
  'nir_pointwise_features_trans_br': 2.357238342775537,
  'rm_pointwise_features_br': 2.248732226812918,
  'rm_cdf_br': 2.506846063492233,
  'rm_featureagg

In [846]:
find_nearest(ss_f.nir_xaxis, center)

1881

In [875]:
raw = ss_f.nir_pointwise_features
xaxis = ss_f.nir_xaxis
temp = SpectralFeatures(xaxis, raw)

temp.generate_bins(4)
temp.generate_bins(5)
temp.generate_bins(10)
temp.generate_bins(20)

plot_key = {}

fig = make_subplots(1,1)

fig.add_trace(go.Scatter(x=temp.energies, y=temp.intensities[0]))

for selected_id in selected_features:

    if selected_id < 16:
        bin_label = 'bin_indices_4'
        poly_label = "const"
        search_party = np.linspace(0,16,4)
    elif selected_id >= 16 and selected_id < 36:
        bin_label = 'bin_indices_5'
        poly_label = "line"
        search_party = np.linspace(16,36,5)
    elif selected_id >= 36 and selected_id < 76:
        bin_label = 'bin_indices_10'
        poly_label = "par"
        search_party = np.linspace(36,76,10)
    elif selected_id >= 76:
        bin_label = 'bin_indices_20' 
        poly_label = "cubic"
        search_party = np.linspace(76,156,20)
    located_bin = np.argwhere(selected_id < search_party)[0][0]
    center = np.median(ss_f.nir_xaxis[getattr(temp, bin_label) == located_bin])
    height_id = find_nearest(ss_f.nir_xaxis, center)
    height = ss_f.nir_pointwise_features[0][height_id]
    plot_key[selected_id] = (center, poly_label, height)
    im = Image.open(f"../app/static/{poly_label}.png")

    fig.add_layout_image(
                dict(
                    source=im,
                    xref="x",
                    yref="y",
                    xanchor="center",
                    yanchor='middle',
                    x=center,
                    y=height,
                    sizex=100,
                    sizey=10,
                    # sizing="contain",
                    opacity=1,
                    layer="above",
                )
            )
fig.show()

In [937]:
nComp = 15
pca = PCA(n_components=nComp)
X = StandardScaler().fit_transform(ss_f.nir_pointwise_features)
pca.fit(X)
mu = np.mean(X, axis=0)

In [952]:
from sklearn.gaussian_process import GaussianProcessRegressor
Xtrain, Xtest = train_test_split(X, test_size=0.25, random_state=10)
gp = GaussianProcessRegressor()
gp.fit(Xtrain, ss_f.br_train)
ypred = gp.predict(Xtest)
root_mean_squared_error(ypred, ss_f.br_test)

16.97637514396001

In [1049]:
ss_f.best_fits['RandomForestRegressor']

#LCEN : 1.4631857124309842, #RF : 1.8510473842137745
ss_f.rm_nir_pointwise_features_br_LCEN['model_params']

[('intercept', 22.370702342612645),
 ('x0', -3.93580754085168e-05),
 ('x1052', 7.288499151864777e-05),
 ('x1993', -0.36460852202903415),
 ('x2212', 0.3872035786856212),
 ('x2213', 0.30505639688240255),
 ('x2313', 0.6198061775899423),
 ('x2378', -0.15717403343557934),
 ('x2379', -0.1688259571229503),
 ('x2380', -0.16551257679797698),
 ('x2382', -0.15845259193247846),
 ('x2392', -0.08971392105607269),
 ('x2581', 0.43169614877159995),
 ('x2680', -0.529354329069838),
 ('x2681', -0.5205199316484405),
 ('x2768', 0.8065148137224322),
 ('x2769', 0.5812792398619591),
 ('x3021', -0.4109655326423094),
 ('x3022', -0.38760396934035585),
 ('x3240', -1.0397599801140538),
 ('x3510', -0.7955988186695395),
 ('x3849', -0.551123873228381),
 ('x4082', 1.5665702097423875),
 ('x4084', 0.5667293596312337),
 ('x4105', -1.1657405018121203),
 ('x4114', 1.4021123909076043)]

In [1048]:
fig = make_subplots(1,2, specs=[[{"secondary_y": True}, {"secondary_y": True}]], 
                    subplot_titles=['LCEN', 'Random Forest'])

m1 = ss_f.rm_pointwise_features[0].ravel()
m2 = ss_f.nir_pointwise_features[0].ravel()
fig.add_trace(go.Scatter(x=np.arange(len(m1)), y=m1, mode='lines'), row=1, col=1) 
fig.add_trace(go.Scatter(x=len(m1) + np.arange(len(m2)), y=m2, mode='lines'), secondary_y=True, row=1, col=1)   

fig.add_trace(go.Scatter(x=np.arange(len(m1)), y=m1, mode='lines'), row=1, col=2) 
fig.add_trace(go.Scatter(x=len(m1) + np.arange(len(m2)), y=m2, mode='lines'), secondary_y=True, row=1, col=2)    

for model, mn, j in zip([ss_f.rm_nir_pointwise_features_br_LCEN, ss_f.rm_nir_pointwise_features_br_RandomForestRegressor], ['LCEN', 'Random Forest'], [0,1]):      
    plotter = pm(fig=fig, xaxis=np.arange(len(m1) + len(m2)), selected_features=model['selected_features'], raw=ss_f.rm_nir_pointwise_features[0].ravel())#.plot_lines()
    xaxis = np.arange(len(m1) + len(m2))
    selected_features = model['selected_features']
    # print(model['score'])
    kwargs = {
                "marker_size": 15,
                "marker_symbol": "line-ns",
                "marker_line_width": 2,
                "mode": "markers",
                "hoverinfo": "none",
                "marker": dict(
                    color=plotter.selected_vals,
                    colorscale="Viridis",
                    showscale=plotter.show_scale,
                    line=dict(width=2, color=plotter.selected_vals, colorscale="Viridis"),
                ),
            }

    for i in np.arange(0, np.max(m2), 1):
        fig.add_trace(
            go.Scatter(
                x=plotter.xaxis[plotter.selected_features],
                y=i*np.ones_like(plotter.selected_features),
                showlegend=False,
                name="mn",
                **kwargs,
            ),
            row=1,
            col=j+1,
            secondary_y=True,
        )

fig.show()               

In [1052]:
rmse = pd.DataFrame(ss_f.best_fits).T[['nir_pointwise_features_br', 'nir_polynomial_featuresall_br', 'nir_featureagg_br', 'nir_pca_br']].loc['LCEN']
rmse

nir_pointwise_features_br        1.561228
nir_polynomial_featuresall_br    1.374171
nir_featureagg_br                1.474640
nir_pca_br                       1.515364
Name: LCEN, dtype: float64

In [1091]:
type(go.Figure)

type

In [1082]:
from spectrascope.plot_spectral_features import PlotModel as pm 
from spectrascope.plot_spectral_features import PlotSpectralFeatures as psf 

spec = 'nir'
mn = 'LCEN'

rmse = pd.DataFrame(ss_f.best_fits).T[['nir_pointwise_features_br', 'nir_polynomial_featuresall_br', 'nir_featureagg_br', 'nir_pca_br']].loc[mn].values

subplot_titles=['Raw Data', 'Polynomial Features', 'Feature Agglomeration', 'PCA']
subplot_titles = [title + f', RMSE={round(100 * r / np.max(br), 2)} %' for title, r in zip(subplot_titles, rmse)]

fig = make_subplots(2,1, specs=[[{"secondary_y": True}], [{"secondary_y": True}]], 
                    subplot_titles=subplot_titles)#, specs=[[{"secondary_y": True}, {"secondary_y": True}]], subplot_titles=['RF Feature Importances', 'LCEN Selected Features'])
            
for i, feat in enumerate(['pointwise_features', 'polynomial_featuresall']):#, 'featureagg', 'pca']):
    model = getattr(ss_f, f'{spec}_{feat}_br_{mn}')

    selected_features = model['selected_features']
    selected_vals = model['selected_vals']
    raw = getattr(ss_f, f'{spec}_{feat}')
    xaxis = getattr(ss_f, f'{spec}_xaxis')

    kwargs_pca = {'feat': raw,
    'mode': 'markers',
    'showlegend': False,
    'marker': {'color': ss_f.ph,
    # 'colorscale': 'Viridis',
    'showscale': True,
    'colorbar': {'title': 'TSS'}}}
        
    # feature_importances = model['P'][0].best_estimator_.feature_importances_

    # feature_importances = model['P'].feature_importances_
    # feature_importances = feature_importances / np.max(feature_importances)
    if feat == 'pointwise_features': 
        
        plotter = pm(selected_features=selected_features, raw=ss_f.nir_pointwise_features[0].ravel(), fig=fig, xaxis=xaxis,row=i+1,col=1, xaxis_label='Wavelength [nm]', yaxis_label='Reflection [%]', selected_vals=None, plot_importances=False)
        plotter.__getattribute__(f'plot_{feat}').__call__()
    elif feat == 'polynomial_featuresall': 
        plotter = pm(selected_features=selected_features, raw=ss_f.nir_pointwise_features[0].ravel(), fig=fig, xaxis=xaxis,row=i+1,col=1, xaxis_label='Wavelength [nm]', yaxis_label='Reflection [%]', selected_vals=selected_vals, plot_importances=False)
        plotter.__getattribute__(f'plot_{feat}').__call__(sizex=200, sizey=50)
    else: 
        plotter = pm(selected_features=selected_features, raw=ss_f.nir_pointwise_features[0].ravel(), fig=fig, xaxis=xaxis,row=(i %2) + 1,col=1, xaxis_label=f'Component {selected_features[0]}', yaxis_label=f'Component {selected_features[1]}', selected_vals=selected_vals, plot_importances=True)
        plotter.__getattribute__(f'plot_{feat}').__call__(**kwargs_pca)
fig.update_layout(title_text=f'Predicting Brixs, French, {mn}')
# fig.update_yaxes(title_text='Reflection [%]', row=1, col=1)

fig.update_layout(
    autosize=False,
    width=500,
    height=1000,
    # margin=dict(
    #     l=50,
    #     r=50,
    #     b=100,
    #     t=100,
    #     pad=4
    # )
)
fig.show()


In [713]:
# temp = SpectraScope(X=([XSF_br, XRF_br]))
# temp.add_x_axis('spec0', nir_wls)
# temp.add_spectral_feature('spec0', 'pointwise_features', xrange=temp.spec0_xaxis)
# temp.add_spectral_feature('spec0', 'polynomial_featuresall')
# ts = TSNE(n_components=3).fit_transform(PCA(n_components=50).fit_transform(temp.spec0_pointwise_features))

ts = embeddings['t-SNE embedding'].fit_transform(StandardScaler().fit_transform(ss_f.rm_pca))
# ts = PCA(n_components=15).fit_transform(temp.spec0_polynomial_featuresall)

fig = px.scatter_3d(x = ts[:,0], y=ts[:,1], z=ts[:,2],
              color=ta)
fig.update_layout(coloraxis_colorbar=dict(title='Ta'))
fig.update_layout(title_text='t-SNE of French NIR Spectra, poly fitting')
fig.show()

In [603]:
X = XRF_br
y = br
n_samples, n_features = X.shape
n_neighbors = 30

In [609]:
import numpy as np
from matplotlib import offsetbox

from sklearn.preprocessing import MinMaxScaler


In [690]:
from sklearn.decomposition import TruncatedSVD
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import RandomTreesEmbedding
from sklearn.manifold import (
    MDS,
    TSNE,
    Isomap,
    LocallyLinearEmbedding,
    SpectralEmbedding,
)
from sklearn.neighbors import NeighborhoodComponentsAnalysis
from sklearn.pipeline import make_pipeline
from sklearn.random_projection import SparseRandomProjection

embeddings = {
    "Random projection embedding": SparseRandomProjection(
        n_components=2, random_state=42
    ),
    "Truncated SVD embedding": TruncatedSVD(n_components=2),
    "Linear Discriminant Analysis embedding": LinearDiscriminantAnalysis(
        n_components=2
    ),
    "Isomap embedding": Isomap(n_neighbors=n_neighbors, n_components=2),
    "Standard LLE embedding": LocallyLinearEmbedding(
        n_neighbors=n_neighbors, n_components=2, method="standard"
    ),
    "Modified LLE embedding": LocallyLinearEmbedding(
        n_neighbors=n_neighbors, n_components=2, method="modified"
    ),
    "Hessian LLE embedding": LocallyLinearEmbedding(
        n_neighbors=n_neighbors, n_components=2, method="hessian"
    ),
    "LTSA LLE embedding": LocallyLinearEmbedding(
        n_neighbors=n_neighbors, n_components=2, method="ltsa"
    ),
    "MDS embedding": MDS(n_components=2, n_init=1, max_iter=120, n_jobs=2),
    "Random Trees embedding": make_pipeline(
        RandomTreesEmbedding(n_estimators=200, max_depth=5, random_state=0),
        TruncatedSVD(n_components=2),
    ),
    "Spectral embedding": SpectralEmbedding(
        n_components=2, random_state=0, eigen_solver="arpack"
    ),
    "t-SNE embedding": TSNE(
        n_components=3,
        max_iter=500,
        n_iter_without_progress=150,
        n_jobs=2,
        random_state=0,
        # init='pca',
        method='exact',
        perplexity=50
    ),
    "NCA embedding": NeighborhoodComponentsAnalysis(
        n_components=2, init="pca", random_state=0
    ),
}

In [653]:
embeddings.items()

dict_items([('Random projection embedding', SparseRandomProjection(n_components=2, random_state=42)), ('Truncated SVD embedding', TruncatedSVD()), ('Linear Discriminant Analysis embedding', LinearDiscriminantAnalysis(n_components=2)), ('Isomap embedding', Isomap(n_neighbors=30)), ('Standard LLE embedding', LocallyLinearEmbedding(n_neighbors=30)), ('Modified LLE embedding', LocallyLinearEmbedding(method='modified', n_neighbors=30)), ('Hessian LLE embedding', LocallyLinearEmbedding(method='hessian', n_neighbors=30)), ('LTSA LLE embedding', LocallyLinearEmbedding(method='ltsa', n_neighbors=30)), ('MDS embedding', MDS(max_iter=120, n_init=1, n_jobs=2)), ('Random Trees embedding', Pipeline(steps=[('randomtreesembedding',
                 RandomTreesEmbedding(n_estimators=200, random_state=0)),
                ('truncatedsvd', TruncatedSVD())])), ('Spectral embedding', SpectralEmbedding(eigen_solver='arpack', random_state=0)), ('t-SNE embedding', TSNE(max_iter=500, method='exact', n_iter_wit

In [654]:
from time import time 
def test_embeddings(X): 
    projections, timing = {}, {}
    for name, transformer in embeddings.items():
        try: 
            if name.startswith("Linear Discriminant Analysis"):
                data = X.copy()
                data.flat[:: X.shape[1] + 1] += 0.01  # Make X invertible
            else:
                data = X

            # print(f"Computing {name}...")
            start_time = time()
            projections[name] = transformer.fit_transform(data, y)
            timing[name] = time() - start_time
        except ValueError:
            pass
    return projections

In [655]:
def plot_embedding(X, title):
    _, ax = plt.subplots()
    X = MinMaxScaler().fit_transform(X)

    for digit in ph:
        ax.scatter(
            *X[y == digit].T,
            marker=f"${digit}$",
            s=60,
            color=plt.cm.Dark2(digit),
            alpha=0.425,
            zorder=2,
        )
    shown_images = np.array([[1.0, 1.0]])  # just something big
    for i in range(X.shape[0]):
        # plot every digit on the embedding
        # show an annotation box for a group of digits
        dist = np.sum((X[i] - shown_images) ** 2, 1)
        if np.min(dist) < 4e-3:
            # don't show points that are too close
            continue
        shown_images = np.concatenate([shown_images, [X[i]]], axis=0)
        # imagebox = offsetbox.AnnotationBbox(
        #     offsetbox.OffsetImage(digits.images[i], cmap=plt.cm.gray_r), X[i]
        # )
        # imagebox.set(zorder=1)
        # ax.add_artist(imagebox)

    ax.set_title(title)
    ax.axis("off")

In [698]:
embeddings['t-SNE embedding']

TSNE(max_iter=500, method='exact', n_components=3, n_iter_without_progress=150,
     n_jobs=2, perplexity=50, random_state=0)

In [711]:
tsne = TSNE(
        n_components=2,
        max_iter=5000,
        n_iter_without_progress=150,
        n_jobs=2,
        # random_state=0,
        # init='pca',
        method='exact',
        perplexity=20,
        init = 'pca'
    )

# pca = PCA(n_components=2)
ts_pt = tsne.fit_transform(ss_f.rm_pointwise_features)
ts_ply = tsne.fit_transform(ss_f.rm_cdf)


fig = make_subplots(1,3, subplot_titles=['RM Pointwise', 'RM CDF', 'NIR_PCA'])
fig.add_trace(go.Scatter(x = ts_pt[:,0], y = ts_pt[:,1], mode='markers',   
                         marker= {'color': ss_f.ph,
                        # 'colorscale': 'Viridis',
                        'showscale': True,
                        'colorbar': {'title': 'ta'}}, showlegend=False), row=1, col=1)

fig.add_trace(go.Scatter(x = ts_ply[:,0], y = ts_ply[:,1], mode='markers',   
                         marker= {'color': ss_f.ph,
                        # 'colorscale': 'Viridis',
                        'showscale': True,
                        'colorbar': {'title': 'ta'}}, showlegend=False), row=1, col=2)

fig.add_trace(go.Scatter(x = ss_f.nir_pca[:,0], y = ss_f.nir_pca[:,1], mode='markers',   
                         marker= {'color': ss_f.ph,
                        # 'colorscale': 'Viridis',
                        'showscale': True,
                        'colorbar': {'title': 'ta'}}, showlegend=False), row=1, col=3)

In [402]:
fig = make_subplots(1,2, specs=[[{"secondary_y": True}, {"secondary_y": True}]], 
                    subplot_titles=['pca', 'featureagg'])#, specs=[[{"secondary_y": True}, {"secondary_y": True}]], subplot_titles=['RF Feature Importances', 'LCEN Selected Features'])

for i, feat in enumerate(['pca', 'featureagg']):
    model = getattr(ss_f, f'rm_{feat}_ph_RandomForestRegressor')

    selected_features = model['selected_features']
    selected_vals = model['selected_vals']
    raw = getattr(ss_f, f'rm_{feat}')
    xaxis = getattr(ss_f, f'rm_xaxis')
    
    kwargs_pca = {'feat': raw,
    'mode': 'markers',
    'marker': {'color': ss_f.ph,
    'colorscale': 'Viridis',
    'showscale': True,
    'colorbar': {'title': 'ph'}}}
    
    # feature_importances = model['P'][0].best_estimator_.feature_importances_

    feature_importances = model['P'].feature_importances_
    feature_importances = feature_importances / np.max(feature_importances)
    plotter = pm(selected_features=selected_features, raw=raw, fig=fig, xaxis=xaxis,row=1,col=i+1, xaxis_label=f'component {selected_features[0]}', yaxis_label = f'component {selected_features[1]}', selected_vals=selected_vals, plot_importances=True)
    plotter.plot_pca(**kwargs_pca)

fig.show()

In [421]:
labels = {str(i): f"PC {i+1}" for i in range(5)}
labels['color'] = 'pH'

fig = px.scatter_matrix(ss_f.rm_featureagg[:,:5], labels=labels, color=ss_f.ph)
fig.update_traces(diagonal_visible=False)
fig.update_coloraxes(colorscale='Viridis')

In [ ]:
ss_f.do_all(feats, 'ph', 0.25, sk_kwargs=[RF_kwargs], do_all_existing=True)

In [ ]:
plt.plot(ss_f.rm_pointwise_features_ph_LCEN['yhat_test_nontrans'])

In [ ]:
ss_f.do_sk('rm_cdf', 'ph', 0.25, **RF_kwargs)
ss_f.do_sk('rm_pca', 'ph', 0.25, **RF_kwargs)

ss_f.do_sk('nir_cdf', 'ph', 0.25, **RF_kwargs)
ss_f.do_sk('nir_pca', 'ph', 0.25, **RF_kwargs)

In [ ]:
lasso_kwargs = {'model':Lasso(), 
                'param_grid':{'alpha':[0.001, 0.01,0.005, 0.05, 0.1,0.2,0.5,0.9],
                            'fit_intercept':[True]
                    },
                'refit':True,
                'metric':mean_squared_error}

In [ ]:
ss_f.do_sk(attr, 'ph', 0.25, **lasso_kwargs)

In [ ]:
from sklearn.metrics import root_mean_squared_error

In [ ]:
rmse_df = {'RandomForestRegressor':{}}
for attr in [attr + '_RandomForestRegressor' for attr in ss_f.best_fits['RandomForestRegressor'].keys()]: 
    ypred = getattr(ss_f, attr)['ypred']
    rmse_df['RandomForestRegressor'][attr.replace('_RandomForestRegressor', '')] = root_mean_squared_error(ss_f.ph_test, ypred) * 100

In [ ]:
sns.heatmap(pd.DataFrame(rmse_df))

In [ ]:
for attr in [f.replace('_ph', '') for f in top_feats]:
    if 'trans' not in attr: 
        ss_f.do_LCEN(attr, 'ph', 0.25, **lcen_kwargs)

In [ ]:
import seaborn as sns 
sns.heatmap(pd.DataFrame(ss_f.best_fits))

In [ ]:
top = 4
top_feats = pd.DataFrame(ss_f.best_fits).sort_values(by='RandomForestRegressor')[:top].index

In [ ]:
f, axs = plt.subplots(2,2, figsize=(10,10), constrained_layout=True)
axs = axs.ravel()
for i, attr in enumerate(top_feats):
    rf_importances = getattr(ss_f, attr + '_RandomForestRegressor')['P'][0].best_estimator_.feature_importances_
    axs[i].plot(rf_importances / np.max(rf_importances))
    axs[i].set_title(attr)
axs[2].plot(ss_f.nir[0] / np.max(ss_f.nir[0]))

In [ ]:
truth_dir = '/Users/amalyajohnson/Desktop/SpectraScope/Datasets/Grape-Chemical-Composition-main/Ground Truth/'
white = 'White Variety.xlsx'
red = 'Red Variety.xlsx'

white = pd.read_excel(truth_dir + white)
red = pd.read_excel(truth_dir + red)

In [ ]:
wl = np.array(data2[name]['wvlength'])

for key in data2[name]['spec']: 
    arr = data2[name]['spec'][key]
    arr = np.array(arr)
    plt.plot(wl, arr[0], label=f'{key}')
    
plt.legend

In [ ]:
shift = data['RAMAN_Data_09082022']['raman_shift']
wl = data['RAMAN_Data_09082022']['wvlength']

shift = np.array(shift)
wl = np.array(wl)

for key in data['RAMAN_Data_09082022']['raman']: 
    for key2 in data['RAMAN_Data_09082022']['raman'][key]: 
        arr = data['RAMAN_Data_09082022']['raman'][key][key2]
        arr = np.array(arr)
        plt.plot(shift, arr[1], label=f'{key} {key2}')
        
plt.legend()